# P2P tensor communication

Use `torch.Tensor` payloads with `flock.torch.send`, `recv`, `isend`, and `irecv`.

In [2]:
# ruff: noqa: F401, F841, I001

import flock
import flock.torch as ft
import torch
from dist_puzzles import check

## 1. Ping-pong

Rank 0 sends a tensor to rank 1. Rank 1 sends back that tensor plus one.

In [4]:
start = torch.tensor([41.0])


@flock.distribute(workers=2)
async def ping_pong(start: torch.Tensor):
    rank = flock.get_rank()
    if rank == 0:
        await ft.send(1, start)
        return await ft.recv(1)

    x = await ft.recv(0)
    await ft.send(0, x + 1)
    return None


check(ping_pong(start), [torch.tensor([42.0]), None])

## 2. Blocking exchange

Two ranks exchange tensors with blocking `send` / `recv` without deadlocking.

In [5]:
values = [torch.tensor([0]), torch.tensor([1])]


@flock.distribute(workers=2)
async def blocking_exchange(x: torch.Tensor):
    rank = flock.get_rank()

    if rank == 0:
        await ft.send(1, x)
        return await ft.recv(1)

    other = await ft.recv(0)
    await ft.send(0, x)
    return other


check(blocking_exchange(flock.per_rank(values)), [torch.tensor([1]), torch.tensor([0])])

## 3. Pair exchange by parity

Ranks exchange tensors with their pair: `0 ↔ 1`, `2 ↔ 3`.

In [6]:
values = [torch.tensor([rank]) for rank in range(4)]


@flock.distribute(workers=4)
async def parity_exchange(x: torch.Tensor):
    rank = flock.get_rank()
    partner = rank ^ 1

    if rank % 2 == 0:
        await ft.send(partner, x)
        return await ft.recv(partner)

    other = await ft.recv(partner)
    await ft.send(partner, x)
    return other


check(
    parity_exchange(flock.per_rank(values)),
    [torch.tensor([1]), torch.tensor([0]), torch.tensor([3]), torch.tensor([2])],
)

## 4. Nonblocking exchange

Use `isend` / `irecv` so every rank can post communication before waiting.

In [7]:
values = [torch.tensor([0]), torch.tensor([1])]


@flock.distribute(workers=2)
async def nonblocking_exchange(x: torch.Tensor):
    rank = flock.get_rank()
    peer = 1 - rank

    send_work = ft.isend(peer, x)
    recv_work = ft.irecv(peer)
    other = await recv_work.wait()
    await send_work.wait()
    return other


check(nonblocking_exchange(flock.per_rank(values)), [torch.tensor([1]), torch.tensor([0])])

## 5. Ring shift

Each rank sends its tensor to the next rank and receives from the previous rank.

In [8]:
values = [torch.tensor([rank]) for rank in range(4)]


@flock.distribute(workers=4)
async def ring_shift(x: torch.Tensor):
    rank = flock.get_rank()
    world = flock.get_world_size()
    dst = (rank + 1) % world
    src = (rank - 1) % world

    await ft.isend(dst, x).wait()
    value = await ft.recv(src)
    return value


check(
    ring_shift(flock.per_rank(values)),
    [torch.tensor([3]), torch.tensor([0]), torch.tensor([1]), torch.tensor([2])],
)

## 6. Ring token

A token starts at rank 0 and moves once around the ring.

In [9]:
initial = torch.tensor([0])


@flock.distribute(workers=4)
async def ring_token(initial: torch.Tensor):
    rank = flock.get_rank()
    world = flock.get_world_size()
    dst = (rank + 1) % world
    src = (rank - 1) % world

    if rank == 0:
        await ft.send(dst, initial)
        token = await ft.recv(src)
        return token + 1

    token = await ft.recv(src)
    token = token + 1
    await ft.send(dst, token)
    return token


check(
    ring_token(initial),
    [torch.tensor([4]), torch.tensor([1]), torch.tensor([2]), torch.tensor([3])],
)

## 7. Tagged messages

Rank 0 sends two tensors to rank 1 with different tags. Rank 1 receives tag `1` before tag `0`.

In [10]:
tag_zero = torch.tensor([0])
tag_one = torch.tensor([1])


@flock.distribute(workers=2)
async def tagged_messages(tag_zero: torch.Tensor, tag_one: torch.Tensor):
    rank = flock.get_rank()
    if rank == 0:
        await ft.isend(1, tag_zero, tag=0).wait()
        await ft.isend(1, tag_one, tag=1).wait()
        return None

    first = await ft.recv(0, tag=1)
    second = await ft.recv(0, tag=0)
    return first, second


check(tagged_messages(tag_zero, tag_one), [None, (torch.tensor([1]), torch.tensor([0]))])

## 8. Neighbor exchange

Each rank receives tensors from its left and right ring neighbors.

In [11]:
values = [torch.tensor([rank]) for rank in range(4)]


@flock.distribute(workers=4)
async def neighbor_exchange(x: torch.Tensor):
    rank = flock.get_rank()
    world = flock.get_world_size()
    left = (rank - 1) % world
    right = (rank + 1) % world

    await ft.isend(left, x, tag=0).wait()
    await ft.isend(right, x, tag=1).wait()
    from_left = await ft.recv(left, tag=1)
    from_right = await ft.recv(right, tag=0)
    return from_left, from_right


check(
    neighbor_exchange(flock.per_rank(values)),
    [
        (torch.tensor([3]), torch.tensor([1])),
        (torch.tensor([0]), torch.tensor([2])),
        (torch.tensor([1]), torch.tensor([3])),
        (torch.tensor([2]), torch.tensor([0])),
    ],
)

## 9. Any-source receive

Ranks 1, 2, and 3 send to rank 0. Rank 0 receives three tensors with `flock.ANY_SOURCE`.

In [12]:
values = [torch.tensor([rank]) for rank in range(4)]


@flock.distribute(workers=4)
async def any_source_gather(x: torch.Tensor):
    rank = flock.get_rank()
    if rank != 0:
        await ft.send(0, x)
        return None

    received = []
    for _ in range(3):
        received.append(await ft.recv(flock.ANY_SOURCE))
    return sorted(value.item() for value in received)


check(any_source_gather(flock.per_rank(values)), [[1, 2, 3], None, None, None])

## 10. Manual gather to root

Every non-root rank sends its tensor to rank 0. Rank 0 returns tensors ordered by rank.

In [11]:
values = [torch.tensor([rank]) for rank in range(4)]


@flock.distribute(workers=4)
async def manual_gather(x: torch.Tensor):
    rank = flock.get_rank()
    if rank == 0:
        gathered = [x]
        for src in range(1, flock.get_world_size()):
            gathered.append(await ft.recv(src))
        return gathered

    await ft.isend(0, x).wait()
    return None


check(
    manual_gather(flock.per_rank(values)),
    [[torch.tensor([0]), torch.tensor([1]), torch.tensor([2]), torch.tensor([3])], None, None, None],
)

## 11. All-to-all

Each rank splits a local tensor into one slice per destination, then receives one slice from every source.

In [12]:
values = [torch.arange(rank * 10, rank * 10 + 4) for rank in range(4)]


@flock.distribute(workers=4, policy=flock.Fifo())
async def all_to_all(x: torch.Tensor):
    rank = flock.get_rank()
    world = flock.get_world_size()

    sends = []
    for dst in range(world):
        sends.append(ft.isend(dst, x[dst : dst + 1]))

    received = []
    for src in range(world):
        received.append(await ft.recv(src))

    for work in sends:
        await work.wait()

    return sorted(received, key=lambda value: value.item())


check(
    all_to_all(flock.per_rank(values)),
    [
        [torch.tensor([0]), torch.tensor([10]), torch.tensor([20]), torch.tensor([30])],
        [torch.tensor([1]), torch.tensor([11]), torch.tensor([21]), torch.tensor([31])],
        [torch.tensor([2]), torch.tensor([12]), torch.tensor([22]), torch.tensor([32])],
        [torch.tensor([3]), torch.tensor([13]), torch.tensor([23]), torch.tensor([33])],
    ],
)

## 12. Pipeline chain

Three microbatches move through the rank chain. Each nonzero rank adds one before forwarding.

In [13]:
microbatch_count = 3


@flock.distribute(workers=4)
async def pipeline_chain(microbatch_count: int):
    rank = flock.get_rank()
    world = flock.get_world_size()
    outputs = []

    for microbatch in range(microbatch_count):
        if rank == 0:
            value = torch.tensor([microbatch])
        else:
            value = await ft.recv(rank - 1)
            value = value + 1

        if rank != world - 1:
            await ft.send(rank + 1, value)

        outputs.append(value)

    return outputs


check(
    pipeline_chain(microbatch_count),
    [
        [torch.tensor([0]), torch.tensor([1]), torch.tensor([2])],
        [torch.tensor([1]), torch.tensor([2]), torch.tensor([3])],
        [torch.tensor([2]), torch.tensor([3]), torch.tensor([4])],
        [torch.tensor([3]), torch.tensor([4]), torch.tensor([5])],
    ],
)

## 13. Max-rank election

Circulate a candidate around the ring and return the elected rank.

In [14]:
values = [torch.tensor([10]), torch.tensor([3]), torch.tensor([2]), torch.tensor([42])]


@flock.distribute(workers=4)
async def max_rank_election(value: torch.Tensor):
    rank = flock.get_rank()
    world = flock.get_world_size()
    dst = (rank + 1) % world
    src = (rank - 1) % world

    if rank == 0:
        await ft.send(dst, value)
        elected = await ft.recv(src)
        await ft.send(dst, elected)
        return elected

    candidate = await ft.recv(src)
    candidate = torch.maximum(candidate, torch.tensor([rank]))
    await ft.send(dst, candidate)
    elected = await ft.recv(src)
    if rank != world - 1:
        await ft.send(dst, elected)
    return elected


check(
    max_rank_election(flock.per_rank(values)),
    [torch.tensor([10]), torch.tensor([10]), torch.tensor([10]), torch.tensor([10])],
)